# МОДУЛЬ 4: ИНЖЕНЕРИЯ ПРИЗНАКОВ И РАБОТА С НЕСТАНДАРТНЫМИ ДАННЫМИ

## Полный учебный материал (теория + код)

---

# ЧАСТЬ 1. ТЕОРЕТИЧЕСКАЯ ЧАСТЬ

---

## 1.1. Что такое Feature Engineering

**Инженерия признаков (Feature Engineering)** — это процесс создания новых признаков (features) из существующих данных, чтобы улучшить качество и точность моделей машинного обучения. Это не просто техническая процедура, а творческий процесс, требующий понимания предметной области.

### Почему это важно

Существует известное выражение в машинном обучении: «Признаки важнее алгоритмов. Хорошие признаки + простая модель = отличный результат. Плохие признаки + сложная модель = плохой результат». Это означает, что даже самая продвинутая модель не сможет показать хороший результат, если данные не подготовлены должным образом. Инвестиции времени в Feature Engineering окупаются значительно большим приростом качества, чем выбор более сложного алгоритма.

### Основные задачи Feature Engineering

**1. Извлечение скрытой информации.** Например, из даты рождения можно извлечь возраст, день недели, является ли день выходным. Из адреса — район, расстояние до центра. Из текста — ключевые слова и их важность.

**2. Повышение нелинейности.** Линейные модели (линейная регрессия, логистическая регрессия) лучше работают с нелинейными признаками. Например, логарифмирование скошенных данных или создание полиномиальных признаков.

**3. Уменьшение размерности.** Удаление шумных или избыточных признаков снижает риск переобучения и ускоряет обучение.

**4. Работа с нестандартными данными.** Тексты, геоданные, временные ряды, изображения и звуки требуют специальных методов обработки, которые превращают «сырые» данные в числовые признаки.

### Классификация методов по типам данных

| Тип данных | Методы обработки | Примеры инструментов |
|------------|------------------|---------------------|
| Числовые | Масштабирование, биннинг, полиномы, логарифмирование | StandardScaler, pd.cut(), PolynomialFeatures, np.log1p() |
| Категориальные | Кодирование (Label, One-Hot, Target), частотное кодирование | OneHotEncoder, TargetEncoder, pd.get_dummies() |
| Текстовые | TF-IDF, Bag of Words, эмбеддинги (Word2Vec, BERT) | TfidfVectorizer, CountVectorizer, transformers |
| Геоданные | Расстояния (Haversine), кластеризация (DBSCAN), геохеши | haversine, DBSCAN, geohash |
| Временные ряды | Лаги, скользящие окна, извлечение частей даты | shift(), rolling(), dt.dayofweek |

### Типы признаков, которые мы создадим в этом модуле

**Текстовые признаки:** из технической документации скважин (TF-IDF). Мы научимся превращать текстовые описания инцидентов в числовые векторы, которые можно подать в модель.

**Геопространственные признаки:** расстояния между скважинами (формула Haversine), кластеры (алгоритм DBSCAN). Мы научимся работать с координатами широты и долготы.

**Временные признаки:** лаги, скользящие средние, извлечение дня недели и месяца из дат. Мы научимся превращать временные ряды в табличные признаки для моделей.

---

## 1.2. Работа с текстовыми данными (NLP)

В реальных проектах (особенно в нефтегазе и IT) часто встречается текст: описания инцидентов, техническая документация, отчёты о бурении, логи ошибок. Мы не можем подать текст напрямую в модель машинного обучения — нужны числовые признаки.

### Bag of Words (BoW) — простейший метод

Метод Bag of Words считает частоту каждого слова в документе. Мы строим словарь всех уникальных слов в корпусе, а затем для каждого документа считаем, сколько раз каждое слово в нём встречается.

**Плюсы:** простота реализации, интерпретируемость.

**Минусы:** не учитывает важность слов. Частые слова типа «и», «в», «the», «a» доминируют, хотя не несут смысла. Кроме того, теряется порядок слов.

### TF-IDF (Term Frequency — Inverse Document Frequency)

TF-IDF учитывает важность слова. Он основан на простой идее: если слово часто встречается в конкретном документе, но редко в других документах корпуса, то оно важно именно для этого документа.

**TF (Term Frequency)** — как часто слово встречается в документе. Обычно нормируется на длину документа.

**IDF (Inverse Document Frequency)** — как редко слово встречается во всех документах корпуса. Вычисляется как логарифм отношения общего числа документов к числу документов, содержащих это слово.

**Формула:**

```
TFIDF(t, d) = TF(t, d) × IDF(t)

IDF(t) = log(N / DF(t))
```

Где N — общее число документов в корпусе, DF(t) — число документов, содержащих слово t.

**Идея:** если слово часто в конкретном документе, но редко в других — оно важно для этого документа. Например, слово «поглощение» может быть редким в целом, но критически важным в описании конкретного инцидента.

**Пример:** Документ 1: «Lost circulation zone detected». Документ 2: «Normal drilling». Слово «circulation» встречается только в документе 1, поэтому его IDF высокий, и TF-IDF для документа 1 будет большим. Слово «drilling» встречается в документе 2, его TF-IDF будет высоким для документа 2.

---

## 1.3. Работа с геопространственными данными

Координаты (широта и долгота) — это не просто числа. Для них нужны специальные методы, потому что Земля — это сфера, а не плоскость.

### Проблема с евклидовым расстоянием

Если просто вычесть координаты двух точек (lat1 - lat2, lon1 - lon2), мы получим расстояние в градусах, но это не имеет физического смысла. Один градус широты на экваторе и на полюсе — это разные расстояния в километрах.

### 1. Расстояние Хаверсина (Haversine)

Формула Хаверсина позволяет рассчитать расстояние между двумя точками на сфере (Земле) с высокой точностью.

**Формула:**

```
a = sin²(Δφ/2) + cos(φ1) × cos(φ2) × sin²(Δλ/2)

c = 2 × atan2(√a, √(1−a))

d = R × c
```

Где R — радиус Земли (6371 км), φ — широта, λ — долгота, Δφ = φ2 − φ1, Δλ = λ2 − λ1.

**Почему это важно:** при работе с геоданными (скважины, месторождения) расстояние до центра или между объектами может быть важным признаком. Например, скважины, расположенные близко друг к другу, могут иметь схожие геологические условия.

### 2. Кластеризация DBSCAN

Алгоритм DBSCAN (Density-Based Spatial Clustering of Applications with Noise) находит плотные скопления точек в пространстве.

**Зачем:** найти аномальные скважины, которые находятся далеко от остальных, или выделить кусты скважин (кластеры). Скважины в одном кластере могут иметь схожие свойства.

**Параметры DBSCAN:**

`eps` — радиус окрестности (в градусах или километрах). Определяет, насколько близко должны быть точки, чтобы считаться соседями.

`min_samples` — минимальное число точек в окрестности для формирования кластера. Защищает от выбросов.

**Как работает:** для каждой точки проверяется, сколько соседей в радиусе eps. Если соседей больше min_samples, точка становится ядром кластера. Все соседи присоединяются к кластеру. Точки, не попавшие ни в один кластер, помечаются как шум (метка -1).

---

## 1.4. Временные ряды как признаки

Любые данные, привязанные ко времени (датчики, продажи, показания каротажа), можно превратить в мощные признаки для моделей машинного обучения.

### 1. Извлечение частей даты

Из столбца datetime можно достать множество полезных признаков:

`dt.dayofweek` — день недели: 0 — понедельник, 6 — воскресенье. Помогает выявить недельную сезонность.

`dt.month` — месяц: от 1 до 12. Помогает выявить годовую сезонность.

`dt.year`, `dt.quarter`, `dt.day`, `dt.dayofyear` — год, квартал, день месяца, день года.

Является ли день выходным: `(dt.dayofweek >= 5).astype(int)`. В выходные данные могут вести себя иначе.

### 2. Лаги (Lags)

Сдвиг значения на предыдущие шаги: `df['value'].shift(1)` сдвигает значения на 1 строку вниз.

**Зачем:** если мы хотим предсказать давление завтра, нам полезно знать давление вчера. Временные ряды обладают автокорреляцией — значения в соседние моменты времени связаны.

**Пример:** если TARGET_POROSITY вчера была 0.25, сегодня, вероятно, будет близка к этому значению.

### 3. Скользящие окна (Rolling Windows)

Среднее, медиана или сумма за последние N шагов: `df['value'].rolling(window=7).mean()`.

**Зачем:** сглаживает шум и показывает тренд. Скользящее среднее за 7 дней убирает дневные колебания и оставляет недельный тренд.

**Другие функции:** `rolling().std()` — волатильность, `rolling().min()`, `rolling().max()`, `rolling().sum()`.

### 4. Разница (Diff)

`df['value'].diff()` — разница между текущим и предыдущим значением. Показывает скорость изменения признака. Если давление резко выросло — это может быть предвестником аварии.

### 5. Заполнение пропусков после создания лагов

После `shift()` и `diff()` в начале ряда появляются NaN. Их нужно заполнить (например, нулями или средним), чтобы модель могла работать.

---

# ЧАСТЬ 2. ОПИСАНИЕ ЛЕКЦИОННОГО ДАТАСЕТА (KGS WELL LOG DATA)

---

## 2.1. Структура и архитектура датасета

**Источник:** Kansas Geological Survey (KGS) — публичный архив данных каротажа скважин.

**Задача:** прогнозирование свойств пласта (например, пористости или литологии) на основе данных геофизических исследований скважин (ГИС).

**Что такое каротаж?** Каротаж — это измерение физических свойств горных пород вдоль ствола скважины с помощью опускаемых приборов. Данные каротажа позволяют определить литологию (тип породы), пористость, насыщенность флюидами (нефть, газ, вода).

### Архитектура данных (типичный набор)

| № | Столбец | Тип | Описание | Возможные значения |
|---|---------|-----|----------|-------------------|
| 1 | DEPTH | float | Глубина замера (в футах или метрах) | 0 – 10000+ |
| 2 | GR | float | Гамма-каротаж (Gamma Ray) | 0 – 200 API |
| 3 | RHOB | float | Плотностной каротаж (Bulk Density) | 1.5 – 3.0 г/см³ |
| 4 | NPHI | float | Нейтронный каротаж (Neutron Porosity) | -0.15 – 0.60 |
| 5 | SP | float | Потенциал самопроизвольной поляризации | -100 – +100 мВ |
| 6 | LAT | float | Широта устья скважины | 37.0 – 40.0 |
| 7 | LON | float | Долгота устья скважины | -102.0 – -94.0 |
| 8 | DATE | datetime | Дата проведения каротажа | 1980 – 2020 |
| 9 | NOTES | text | Текстовое описание инцидентов | Свободный текст |

### Описание признаков

**DEPTH** — глубина замера. С увеличением глубины меняются давление и температура, что влияет на свойства пород.

**GR (Gamma Ray)** — гамма-каротаж. Измеряет естественную радиоактивность пород. Высокие значения — глины, низкие — песчаники и карбонаты.

**RHOB (Bulk Density)** — плотностной каротаж. Измеряет плотность породы. Позволяет вычислить пористость.

**NPHI (Neutron Porosity)** — нейтронный каротаж. Измеряет водородосодержание, что связано с пористостью.

**SP (Spontaneous Potential)** — потенциал самопроизвольной поляризации. Помогает определить проницаемость пластов.

**LAT, LON** — координаты устья скважины. Нужны для геопространственного анализа.

**DATE** — дата каротажа. Нужна для временного анализа.

**NOTES** — текстовое описание. Содержит информацию об инцидентах (поглощения, газопроявления).

### Особенности датасета

**Пропуски:** данные ГИС часто имеют пропуски из-за отказов приборов, обрывов кабеля.

**Выбросы:** резкие скачки при смене пластов, аномальные значения из-за шума.

**Мультиколлинеарность:** RHOB и NPHI часто коррелируют, так как оба связаны с пористостью.

**Текстовые данные:** в столбце NOTES содержится информация о проблемах при бурении.

---

# ЧАСТЬ 3. ПОЛНЫЙ РАЗБОР КОДА

---

## 3.1. Импорт библиотек

```python
# ============================================================
# ИМПОРТ БИБЛИОТЕК
# ============================================================

import numpy as np
# NumPy: численные операции, массивы, математические функции (sin, cos, radians, arctan2, sqrt).
# Почему: массивы NumPy быстрее списков Python в 10-100 раз.
# Где используется: тригонометрия для Haversine, генерация синтетических данных, математические операции.

import pandas as pd
# Pandas: работа с табличными данными, группировка, скользящие окна, обработка текста.
# Почему: идеальный инструмент для табличных данных и временных рядов.
# Где используется: загрузка данных, создание признаков, объединение датафреймов.

import matplotlib.pyplot as plt
import seaborn as sns
# Визуализация: графики распределений, карты корреляций, временные ряды.
# Почему: помогает понять данные и проверить результаты.

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
# TfidfVectorizer: преобразует текст в матрицу TF-IDF признаков.
# CountVectorizer: Bag of Words (частотный).
# Почему: стандартные методы обработки текста в sklearn.
# Где используется: превращение столбца NOTES в числовые признаки.

from sklearn.cluster import DBSCAN
# DBSCAN: кластеризация на основе плотности для геоданных.
# Почему: не требует задавать число кластеров, находит аномалии.
# Где используется: нахождение кустов скважин по координатам LAT, LON.

from sklearn.preprocessing import StandardScaler
# StandardScaler: масштабирование признаков (z-score).
# Почему: текстовые и геопризнаки имеют разный масштаб.
# Где используется: нормализация всех признаков перед обучением.

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
# Модели и метрики: Random Forest для регрессии (предсказание пористости).
# train_test_split: разделение данных на train/test.
# mean_squared_error, r2_score: метрики качества регрессии.

import warnings
warnings.filterwarnings('ignore')
# Скрываем предупреждения для чистоты вывода.
# В продакшене так делать нельзя — предупреждения надо исправлять.

print("Библиотеки успешно импортированы!")
```

---

## 3.2. Загрузка и создание синтетических данных

```python
# ============================================================
# ЗАГРУЗКА ДАННЫХ (СИМУЛЯЦИЯ KGS)
# ============================================================
# Для примера создадим синтетический датасет, имитирующий структуру KGS.
# В реальном проекте данные загружались бы из CSV или базы данных.

np.random.seed(42)
# Фиксируем seed для воспроизводимости.
# Почему: чтобы при повторном запуске результаты были одинаковыми.

n_samples = 1000
# Количество строк в датасете.

# --- Создаём датафрейм ---
df = pd.DataFrame({
    'WELL_ID': np.random.choice(['W001', 'W002', 'W003', 'W004', 'W005'], n_samples),
    # Идентификатор скважины. Случайный выбор из 5 скважин.

    'DEPTH': np.linspace(1000, 2000, n_samples),
    # Глубина замера. Линейно от 1000 до 2000.

    'GR': np.random.normal(50, 15, n_samples),
    # Гамма-каротаж. Нормальное распределение со средним 50 и std 15.

    'RHOB': np.random.normal(2.3, 0.2, n_samples),
    # Плотность. Среднее 2.3 г/см³, std 0.2.

    'NPHI': np.random.normal(0.25, 0.05, n_samples),
    # Пористость. Среднее 0.25, std 0.05.

    'LAT': np.random.uniform(37.0, 40.0, n_samples),
    # Широта. Равномерное распределение от 37 до 40.

    'LON': np.random.uniform(-102.0, -94.0, n_samples),
    # Долгота. Равномерное распределение от -102 до -94.

    'DATE': pd.date_range(start='2020-01-01', periods=n_samples, freq='D'),
    # Дата. Последовательность дней с 1 января 2020.

    'NOTES': np.random.choice(
        ['Normal drilling', 'Lost circulation zone detected',
         'Gas kick observed', 'No issues',
         'High pressure anomaly', 'Stuck pipe'],
        n_samples
    )
    # Текстовое описание. Случайный выбор из 6 вариантов.
})

# --- Целевая переменная (Пористость) ---
# Формула: Пористость зависит от NPHI, GR и глубины + шум.
df['TARGET_POROSITY'] = (
    0.4 * df['NPHI'] - 0.001 * df['GR'] + 0.0001 * df['DEPTH']
    + np.random.normal(0, 0.01, n_samples)
)
# NPHI (нейтронная пористость) — прямой предиктор.
# GR (гамма-каротаж) — обратная зависимость (глины уменьшают пористость).
# DEPTH — небольшая положительная зависимость.
# Шум — случайность, чтобы модель не была идеальной.

print(f"Размер датасета: {df.shape}")
print(df.head())
```

---

## 3.3. Текстовые признаки (TF-IDF)

```python
# ============================================================
# ИЗВЛЕЧЕНИЕ ПРИЗНАКОВ ИЗ ТЕКСТА (NLP)
# ============================================================

# --- Инициализируем TF-IDF ---
tfidf = TfidfVectorizer(
    max_features=10,
    # Оставляем только 10 самых важных слов.
    # Почему: ограничиваем размерность, чтобы не переобучиться.

    stop_words='english',
    # Удаляем английские стоп-слова (the, a, is, and...).
    # Почему: они не несут смысла.

    ngram_range=(1, 2)
    # Используем 1-граммы (слова) и 2-граммы (пары слов).
    # Почему: ловим фразы "lost circulation", которые важнее отдельных слов.
)

# --- Обучаем и трансформируем текст ---
tfidf_matrix = tfidf.fit_transform(df['NOTES'])
# fit_transform: сначала изучает словарь (fit), потом преобразует (transform).
# Возвращает разреженную матрицу (sparse matrix).

# --- Преобразуем в DataFrame ---
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray(),
    # .toarray() превращает разреженную матрицу в плотную (numpy array).
    # Почему: для объединения с DataFrame нужен плотный массив.

    columns=tfidf.get_feature_names_out()
    # Имена столбцов — это сами слова.
    # get_feature_names_out() возвращает список слов.
)

# --- Соединяем с основным датафреймом ---
df = pd.concat([df.reset_index(drop=True), tfidf_df], axis=1)
# pd.concat — объединение по столбцам (axis=1).
# reset_index(drop=True) — сброс индекса, чтобы индексы совпадали.

print("Признаки из текста:")
print(tfidf_df.head())
print(f"Созданные столбцы: {tfidf.get_feature_names_out()}")
```

---

## 3.4. Геопризнаки (Haversine и DBSCAN)

```python
# ============================================================
# ГЕОПРОСТРАНСТВЕННЫЕ ПРИЗНАКИ
# ============================================================

# --- 1. Расстояние Хаверсина (до условного центра) ---
def haversine_distance(lat1, lon1, lat2, lon2):
    """Рассчитывает расстояние между двумя точками на Земле в км."""
    R = 6371.0
    # Радиус Земли в км.

    # Перевод градусов в радианы
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    # map применяет функцию к каждому элементу списка.
    # np.radians переводит градусы в радианы.

    # Формула Хаверсина
    dlat = lat2 - lat1
    # Разница широт.

    dlon = lon2 - lon1
    # Разница долгот.

    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    # Центральная часть формулы.

    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    # Арктангенс для получения угла.

    return R * c
    # Умножаем на радиус Земли, получаем расстояние в км.

# Центр (например, главный офис)
CENTER_LAT, CENTER_LON = 38.5, -98.0

df['DIST_FROM_CENTER'] = haversine_distance(
    df['LAT'], df['LON'], CENTER_LAT, CENTER_LON
)
# Почему: расстояние до центра может коррелировать с типом геологии.

# --- 2. Кластеризация DBSCAN ---
dbscan = DBSCAN(
    eps=0.5,
    # Радиус окрестности (в градусах).
    # 0.5 градуса ≈ 55 км.

    min_samples=5
    # Минимум 5 точек для кластера.
    # Защита от одиночных выбросов.
)

# Обучаем DBSCAN на координатах
df['CLUSTER'] = dbscan.fit_predict(df[['LAT', 'LON']])
# fit_predict возвращает метки кластеров.
# -1 = шум/аномалия.

print("Кластеры скважин:")
print(df['CLUSTER'].value_counts())
# -1 — это выбросы (скважины далеко от кустов).
```

---

## 3.5. Временные признаки

```python
# ============================================================
# ВРЕМЕННЫЕ РЯДЫ КАК ПРИЗНАКИ
# ============================================================

# --- 1. Извлечение частей даты ---
df['YEAR'] = df['DATE'].dt.year
# Год.

df['MONTH'] = df['DATE'].dt.month
# Месяц (1-12).

df['DAY_OF_WEEK'] = df['DATE'].dt.dayofweek
# День недели (0-6).

df['IS_WEEKEND'] = (df['DAY_OF_WEEK'] >= 5).astype(int)
# 1 если выходной, 0 иначе.
# Почему: сезонность и выходные могут влиять на данные каротажа.

# --- 2. Лаги (Lags) ---
df['POROSITY_LAG_1'] = df['TARGET_POROSITY'].shift(1)
# Сдвиг на 1 шаг назад.

df['POROSITY_LAG_2'] = df['TARGET_POROSITY'].shift(2)
# Сдвиг на 2 шага назад.
# Почему: если вчера была высокая пористость, сегодня тоже может быть.

# --- 3. Скользящее среднее (Rolling Mean) ---
df['POROSITY_ROLLING_MEAN_7'] = df['TARGET_POROSITY'].rolling(window=7).mean()
# Среднее за последние 7 шагов.
# Почему window=7: сглаживаем шум за неделю.

# --- 4. Разница (Diff) ---
df['POROSITY_DIFF'] = df['TARGET_POROSITY'].diff()
# Разница между текущим и предыдущим значением.
# Почему: скорость изменения признака.

# Заполняем образовавшиеся NaN (после shift/diff)
df = df.fillna(0)
# Почему: модели не умеют работать с NaN.

print("Временные признаки созданы!")
print(df[['DATE', 'TARGET_POROSITY', 'POROSITY_LAG_1', 'POROSITY_ROLLING_MEAN_7']].head(10))
```

---

## 3.6. Объединение признаков и обучение модели

```python
# ============================================================
# ОБЪЕДИНЕНИЕ ПРИЗНАКОВ И ОБУЧЕНИЕ
# ============================================================

# --- Собираем все признаки ---
# Убираем исходные данные, которые уже превратили в признаки
feature_cols = [
    'GR', 'RHOB', 'NPHI', 'DEPTH',
    # Базовые признаки.

    'DIST_FROM_CENTER', 'CLUSTER',
    # Геопризнаки.

    'YEAR', 'MONTH', 'DAY_OF_WEEK', 'IS_WEEKEND',
    # Временные признаки.

    'POROSITY_LAG_1', 'POROSITY_LAG_2',
    # Лаги.

    'POROSITY_ROLLING_MEAN_7', 'POROSITY_DIFF'
    # Скользящее окно и разница.
]

# Добавляем текстовые столбцы (слова из TF-IDF)
feature_cols += list(tfidf.get_feature_names_out())

X = df[feature_cols]
# Матрица признаков.

y = df['TARGET_POROSITY']
# Целевая переменная.

# --- Разделение данных ---
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# 80% train, 20% test.

# --- Масштабирование ---
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# Почему: Random Forest не требует масштабирования, но другие модели — да.
# Делаем для универсальности.

# --- Обучение Random Forest ---
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

# --- Оценка ---
y_pred = model.predict(X_test_scaled)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

# --- Важность признаков ---
importances = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print("Топ-10 важных признаков:")
print(importances.head(10).to_string(index=False))
```

---

## 3.7. Альтернативный вариант — реальная загрузка данных KGS

Если вы работаете с реальным архивом KGS, код загрузки будет выглядеть примерно так:

```python
# ============================================================
# АЛЬТЕРНАТИВА: ЗАГРУЗКА РЕАЛЬНЫХ ДАННЫХ KGS
# ============================================================

# Реальные данные KGS доступны на сайте Kansas Geological Survey:
# https://www.kgs.ku.edu/Magellan/Qualified/index.html
# Формат: LAS-файлы (Log ASCII Standard).

# Для работы с LAS-файлами используется библиотека lasio:
# !pip install lasio

import lasio

# Загрузка одного LAS-файла
las = lasio.read('path/to/well.las')

# Преобразование в DataFrame
df_real = las.df().reset_index()
# .reset_index() — глубина (DEPTH) становится обычным столбцом.
# Результат: DataFrame со столбцами DEPTH, GR, RHOB, NPHI, SP и т.д.

print(df_real.head())
print(df_real.describe())

# Дальнейшая обработка идентична синтетическому примеру:
# - Обработка пропусков (интерполяция по глубине).
# - Удаление выбросов (IQR).
# - Feature Engineering (лаги, скользящие окна).
# - Обучение модели.
```

---

# ЧАСТЬ 4. ИТОГОВЫЙ СПИСОК ОПЕРАТОРОВ

---

## 4.1. Pandas

| Оператор | Назначение |
|----------|------------|
| pd.concat() | Объединение DataFrame (текстовые + основные признаки) |
| df.shift(n) | Сдвиг значений (лаги) |
| df.rolling(window).mean() | Скользящее среднее |
| df.diff() | Разница между текущим и предыдущим значением |
| df.dt.dayofweek | Извлечение дня недели |
| df.dt.month | Извлечение месяца |
| df.dt.year | Извлечение года |
| df.fillna() | Заполнение пропусков (после лагов) |
| df.reset_index() | Сброс индекса (важно при конкатенации) |
| df.value_counts() | Частоты уникальных значений |

## 4.2. Scikit-learn

| Оператор | Назначение |
|----------|------------|
| TfidfVectorizer() | Преобразование текста в TF-IDF матрицу |
| CountVectorizer() | Bag of Words |
| DBSCAN() | Кластеризация геоданных |
| StandardScaler() | Масштабирование признаков |
| RandomForestRegressor() | Модель для прогноза |
| train_test_split() | Разделение на train/test |
| mean_squared_error() | MSE (для RMSE) |
| r2_score() | R² |

## 4.3. NumPy

| Оператор | Назначение |
|----------|------------|
| np.radians() | Перевод градусов в радианы |
| np.sin(), np.cos() | Тригонометрия для Haversine |
| np.arctan2() | Арктангенс для Haversine |
| np.sqrt() | Корень квадратный |
| np.random.normal() | Генерация нормального распределения |
| np.random.uniform() | Генерация равномерного распределения |
| np.linspace() | Линейная последовательность |
| np.random.choice() | Случайный выбор |
| np.random.seed() | Фиксация seed |

## 4.4. Встроенные функции Python

| Оператор | Назначение |
|----------|------------|
| map() | Применение функции к списку (радианы) |
| zip() | Объединение итераторов |
| list() | Преобразование в список |
| range() | Диапазон чисел |

---

# ЧАСТЬ 5. ВЫВОДЫ ПО МОДУЛЮ

---

## 5.1. Что мы освоили

**Текстовые данные.** Превратили описания инцидентов в числовые признаки через TF-IDF. Научились работать с разреженными матрицами и объединять их с табличными данными.

**Геоданные.** Рассчитали расстояние Хаверсина и нашли кластеры скважин через DBSCAN. Научились работать с координатами и находить аномалии.

**Временные ряды.** Создали лаги, скользящие средние и извлекли части дат. Научились превращать временные ряды в табличные признаки.

**Интеграция.** Объединили все признаки в единый датасет и обучили модель. Увидели, как разные типы признаков влияют на важность.

---

## 5.2. Ключевые инсайты

**Feature Engineering даёт больший прирост качества, чем смена модели.** Хорошие признаки + простая модель часто лучше, чем плохие признаки + сложная модель.

**DBSCAN эффективно находит аномалии (выбросы) без учителя.** Метка -1 означает шум — такие объекты могут быть выбросами или требовать отдельного анализа.

**TF-IDF позволяет использовать текстовые логи (NOTES) для прогноза технических параметров.** Даже небольшой текст может дать полезные признаки.

**Лаги и скользящие окна критичны для временных рядов.** Они позволяют модели учитывать историю и тренды.

---

## 5.3. Что дальше изучать

**Target Encoding** для категориальных признаков с высокой кардинальностью. Позволяет избежать взрыва размерности при One-Hot Encoding.

**Эмбеддинги (Word2Vec, BERT)** для более глубокой обработки текста. Позволяют улавливать семантический смысл слов.

**Кросс-валидация по группам (GroupKFold)** для геоданных. Чтобы одна скважина не была в train и test одновременно, иначе модель «подсмотрит» ответ.